# Concurrency

Python offers three distinct concurrency models, each suited to a different class of problem. Choosing the wrong one is one of the most common performance mistakes Python developers make.

## What You'll Learn
1. Concurrency vs parallelism — the fundamental distinction
2. The GIL — what it is, what it protects, and when it matters
3. `threading` — I/O-bound concurrency with threads
4. Thread synchronisation — `Lock`, `RLock`, `Semaphore`, `Event`, `Condition`, `Barrier`
5. `concurrent.futures.ThreadPoolExecutor` — high-level thread pools
6. `multiprocessing` — true CPU parallelism
7. `concurrent.futures.ProcessPoolExecutor` — high-level process pools
8. Shared state between processes — `Manager`, `Value`, `Array`
9. `asyncio` — cooperative async I/O
10. `async`/`await` patterns — tasks, gather, timeouts, queues
11. Mixing concurrency models
12. Free-threaded Python — PEP 703, Python 3.13t & 3.14
13. Thread safety in free-threaded Python — atomicity and new primitives
14. Choosing the right model
15. Common pitfalls and debugging

---

## 1. Concurrency vs Parallelism

These terms are often confused but describe fundamentally different things.

In [ ]:
# The mental model:
#
# CONCURRENCY  — dealing with multiple things at once (structure)
#   One chef juggling multiple dishes by switching attention between them.
#   Only one thing happens at any instant, but progress is made on all.
#   Best for: I/O-bound work (network, disk, database) — tasks spend
#   most of their time WAITING, so switching to another task is cheap.
#
# PARALLELISM  — doing multiple things at once (execution)
#   Multiple chefs each cooking a different dish simultaneously.
#   Multiple things literally happen at the same clock instant.
#   Best for: CPU-bound work (number crunching, image processing)
#   where tasks are always busy and need more raw compute power.
#
#                 ┌──────────────────────────────────────┐
#                 │           TASK TYPE                  │
#                 │    I/O-bound    │   CPU-bound         │
#  ───────────────┼─────────────────┼─────────────────── │
#  threading      │      ✅ Good    │   ❌ GIL blocks     │
#  asyncio        │      ✅ Best    │   ❌ single-thread  │
#  multiprocessing│      ⚠️  Overkill│   ✅ Best          │
#  free-threading │      ✅ Good    │   ✅ Good (3.13t+)  │
#                 └──────────────────────────────────────┘

print("Concurrency model decision guide printed above.")

---

## 2. The GIL — Global Interpreter Lock

In [ ]:
# The GIL is a mutex in CPython that ensures only ONE thread executes
# Python bytecode at a time — even on a multi-core machine.
#
# WHY DOES IT EXIST?
#   CPython's memory management (reference counting) is NOT thread-safe.
#   The GIL is the simple fix: just prevent concurrent Python execution.
#   Benefits:
#     - Simple, fast single-threaded code
#     - C extensions are easy to write (can assume GIL protects objects)
#     - No data races on Python objects in most cases
#
# WHEN DOES THE GIL GET RELEASED?
#   The GIL IS released during:
#     - Blocking I/O (file read, network call, sleep)
#     - Many C extension calls (NumPy, etc. release the GIL for computation)
#     - Every 100 bytecode instructions (sys.getswitchinterval() = 0.005s)
#   This is WHY threading helps for I/O-bound work — the GIL is released
#   while one thread waits, letting another thread run.
#
# WHEN IS THE GIL A BOTTLENECK?
#   CPU-bound Python threads — only one thread runs Python at a time.
#   Adding more threads won't make CPU-bound Python faster.

import sys
print(f"Switch interval: {sys.getswitchinterval()}s")  # 0.005s default

# Check if running in free-threaded mode (3.13+)
import sysconfig
is_free_threaded = bool(sysconfig.get_config_var('Py_GIL_DISABLED'))
print(f"Free-threaded build: {is_free_threaded}")

In [ ]:
# Demonstrating the GIL's effect on CPU-bound threads
import threading, time

def cpu_bound(n: int) -> int:
    """Pure Python CPU work — keeps the GIL held."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 5_000_000

# Sequential — one call after another
t0 = time.perf_counter()
cpu_bound(N)
cpu_bound(N)
seq_time = time.perf_counter() - t0

# Threaded — two threads running "simultaneously"
t0 = time.perf_counter()
t1 = threading.Thread(target=cpu_bound, args=(N,))
t2 = threading.Thread(target=cpu_bound, args=(N,))
t1.start(); t2.start()
t1.join();  t2.join()
thr_time = time.perf_counter() - t0

print(f"Sequential:  {seq_time:.3f}s")
print(f"2 Threads:   {thr_time:.3f}s")
print(f"Speedup:     {seq_time/thr_time:.2f}x  (expected ~1.0x due to GIL)")
print("Threading gives NO speedup for CPU-bound Python code (with standard GIL build).")

---

## 3. `threading` — I/O-Bound Concurrency

In [ ]:
import threading
import time

# Creating and starting threads
def worker(name: str, duration: float) -> None:
    print(f"  [{name}] started")
    time.sleep(duration)   # simulates I/O wait — GIL is released!
    print(f"  [{name}] done after {duration}s")


print("Sequential:")
t0 = time.perf_counter()
worker('A', 0.3)
worker('B', 0.3)
worker('C', 0.3)
print(f"  Total: {time.perf_counter()-t0:.2f}s\n")

print("Threaded (all run concurrently):")
t0 = time.perf_counter()
threads = [
    threading.Thread(target=worker, args=(name, 0.3))
    for name in ['A', 'B', 'C']
]
for t in threads: t.start()
for t in threads: t.join()   # wait for all to complete
print(f"  Total: {time.perf_counter()-t0:.2f}s  (concurrent — ~0.3s vs 0.9s)")

In [ ]:
# Thread creation options
results = {}

# 1. Function-based thread
def fetch(url: str) -> str:
    time.sleep(0.1)   # simulated I/O
    return f"data from {url}"

def threaded_fetch(url: str) -> None:
    results[url] = fetch(url)

t = threading.Thread(target=threaded_fetch, args=('http://example.com',),
                     name='fetch-example', daemon=True)
t.start()
t.join()

# 2. Thread subclass
class WorkerThread(threading.Thread):
    def __init__(self, task_id: int):
        super().__init__(name=f"worker-{task_id}", daemon=True)
        self.task_id = task_id
        self.result  = None

    def run(self) -> None:   # override run() — called by start()
        time.sleep(0.05)
        self.result = self.task_id ** 2


workers = [WorkerThread(i) for i in range(5)]
for w in workers: w.start()
for w in workers: w.join()

print("Results:", [w.result for w in workers])

# Thread properties
t2 = threading.Thread(target=time.sleep, args=(100,), daemon=True)
t2.start()
print(f"\nThread name:    {t2.name}")
print(f"Thread ident:   {t2.ident}")
print(f"Thread alive:   {t2.is_alive()}")
print(f"Is daemon:      {t2.daemon}")

# Active threads
print(f"Active threads: {threading.active_count()}")
print(f"Current thread: {threading.current_thread().name}")

---

## 4. Thread Synchronisation Primitives

In [ ]:
import threading

# ── Lock — mutual exclusion ───────────────────────────────────────────────
# Prevents multiple threads modifying shared state simultaneously.

# ❌ Race condition — unsynchronised counter
counter = 0

def unsafe_increment(n):
    global counter
    for _ in range(n):
        counter += 1   # not atomic: read, add, write — can interleave!

threads = [threading.Thread(target=unsafe_increment, args=(10000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Unsafe counter (expected 50000): {counter}")

# ✅ Thread-safe with a Lock
safe_counter = 0
lock = threading.Lock()

def safe_increment(n):
    global safe_counter
    for _ in range(n):
        with lock:           # acquire on enter, release on exit (even on exception)
            safe_counter += 1

threads = [threading.Thread(target=safe_increment, args=(10000,)) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe counter   (expected 50000): {safe_counter}")

In [ ]:
# ── RLock — reentrant lock ────────────────────────────────────────────────
# A thread that already holds the lock can acquire it again.
# Needed when a locked function calls another locked function.

rlock = threading.RLock()

def outer():
    with rlock:
        print("  outer acquired")
        inner()   # also tries to acquire the same lock
        print("  outer released")

def inner():
    with rlock:   # works because same thread can re-acquire RLock
        print("  inner acquired")

outer()

In [ ]:
# ── Semaphore — limit concurrent access ──────────────────────────────────
# Allows at most N threads to hold the semaphore simultaneously.
# Classic use: connection pool, rate limiting.

import time, threading

MAX_CONNECTIONS = 3
sem = threading.Semaphore(MAX_CONNECTIONS)

def use_connection(worker_id: int) -> None:
    with sem:   # blocks if 3 connections already active
        print(f"  Worker {worker_id}: connected")
        time.sleep(0.2)
        print(f"  Worker {worker_id}: disconnected")

print("Connection pool (max 3 concurrent):")
threads = [threading.Thread(target=use_connection, args=(i,)) for i in range(7)]
for t in threads: t.start()
for t in threads: t.join()

In [ ]:
# ── Event — signal between threads ───────────────────────────────────────
# One thread signals (sets), other threads wait for it.

ready_event = threading.Event()

def producer():
    print("  Producer: preparing data...")
    time.sleep(0.3)
    print("  Producer: data ready, signalling")
    ready_event.set()   # unblocks all waiting threads

def consumer(cid: int):
    print(f"  Consumer {cid}: waiting for data...")
    ready_event.wait()  # blocks until event is set
    print(f"  Consumer {cid}: processing data")

threads = [
    threading.Thread(target=producer),
    threading.Thread(target=consumer, args=(1,)),
    threading.Thread(target=consumer, args=(2,)),
]
for t in threads: t.start()
for t in threads: t.join()

In [ ]:
# ── Condition — wait for a specific state ────────────────────────────────
# More powerful than Event: threads can wait for a condition tied to
# shared state, re-check it after being notified, and only proceed when true.

import queue as queue_module

condition = threading.Condition()
shared_queue = []

def producer_cond():
    for i in range(5):
        time.sleep(0.1)
        with condition:
            shared_queue.append(i)
            print(f"  Produced: {i}")
            condition.notify()   # wake one waiting consumer

def consumer_cond():
    consumed = 0
    while consumed < 5:
        with condition:
            while not shared_queue:        # spurious wakeup guard
                condition.wait()           # releases lock, waits for notify
            item = shared_queue.pop(0)
        print(f"  Consumed: {item}")
        consumed += 1

p = threading.Thread(target=producer_cond)
c = threading.Thread(target=consumer_cond)
c.start(); p.start()
p.join();  c.join()

In [ ]:
# ── Barrier — synchronise a group of threads ─────────────────────────────
# All threads block until every member has reached the barrier, then proceed.

THREADS = 4
barrier = threading.Barrier(THREADS)

def phase_worker(wid: int):
    # Phase 1 — different durations
    time.sleep(0.1 * wid)
    print(f"  Worker {wid}: phase 1 done, waiting at barrier")
    barrier.wait()   # blocks until all 4 reach here
    # All threads now proceed together
    print(f"  Worker {wid}: phase 2 started")

threads = [threading.Thread(target=phase_worker, args=(i,)) for i in range(1, THREADS+1)]
for t in threads: t.start()
for t in threads: t.join()

# ── thread-local storage ──────────────────────────────────────────────────
# Each thread has its own copy of a thread-local variable.
local = threading.local()

def set_and_read(value):
    local.data = value           # each thread's own copy
    time.sleep(0.01)
    print(f"  Thread {value}: local.data = {local.data}")

print("\nThread-local storage:")
threads = [threading.Thread(target=set_and_read, args=(i,)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()

In [ ]:
# ── Thread-safe queue — the preferred inter-thread communication pattern ──
# queue.Queue is fully thread-safe. Prefer it over manual Lock + list.

import queue

task_queue = queue.Queue(maxsize=10)

def producer_queue():
    for i in range(8):
        task_queue.put(i)          # blocks if queue is full
        print(f"  Put: {i}")
    task_queue.put(None)           # poison pill — tells consumer to stop

def consumer_queue():
    while True:
        item = task_queue.get()    # blocks until an item is available
        if item is None:
            break
        print(f"  Got: {item}")
        task_queue.task_done()     # signal that item was processed

p = threading.Thread(target=producer_queue)
c = threading.Thread(target=consumer_queue)
p.start(); c.start()
p.join();  c.join()

# queue.Queue variants:
# queue.LifoQueue  — stack (last in, first out)
# queue.PriorityQueue — items sorted by priority (min first)
# queue.SimpleQueue — simpler, no size limit, no task_done/join

---

## 5. `concurrent.futures.ThreadPoolExecutor`

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed, Future
import time, urllib.request

def simulate_api_call(endpoint: str, delay: float) -> dict:
    """Simulate a slow API call."""
    time.sleep(delay)
    return {'endpoint': endpoint, 'status': 200, 'delay': delay}

endpoints = [
    ('/users',    0.3),
    ('/orders',   0.5),
    ('/products', 0.2),
    ('/reviews',  0.4),
    ('/search',   0.6),
]

# ── map() — simplest pattern ──────────────────────────────────────────────
print("Using map():")
t0 = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as pool:
    # map() returns results in submission ORDER (not completion order)
    results = list(pool.map(
        lambda args: simulate_api_call(*args),
        endpoints
    ))

print(f"  All done in {time.perf_counter()-t0:.2f}s (max delay was 0.6s)")
for r in results:
    print(f"  {r['endpoint']}: {r['status']} ({r['delay']}s)")

In [ ]:
# ── submit() + as_completed() — process results as they arrive ───────────
print("Using submit() + as_completed():")
t0 = time.perf_counter()

with ThreadPoolExecutor(max_workers=5) as pool:
    # submit() returns a Future immediately
    future_to_endpoint = {
        pool.submit(simulate_api_call, ep, delay): ep
        for ep, delay in endpoints
    }

    # as_completed yields futures in COMPLETION order (fastest first)
    for future in as_completed(future_to_endpoint):
        endpoint = future_to_endpoint[future]
        try:
            result = future.result()   # raises if the callable raised
            elapsed = time.perf_counter() - t0
            print(f"  {elapsed:.2f}s  {endpoint}: done")
        except Exception as e:
            print(f"  {endpoint}: FAILED — {e}")

In [ ]:
# ── Future API ────────────────────────────────────────────────────────────
with ThreadPoolExecutor(max_workers=2) as pool:
    f: Future = pool.submit(simulate_api_call, '/health', 0.2)

    print(f"running:   {f.running()}")
    print(f"done:      {f.done()}")

    result = f.result(timeout=5.0)   # blocks; raises TimeoutError if too slow
    print(f"result:    {result}")
    print(f"done now:  {f.done()}")

# Add callback to run when future completes
with ThreadPoolExecutor() as pool:
    f = pool.submit(simulate_api_call, '/callback-test', 0.1)
    f.add_done_callback(
        lambda fut: print(f"  Callback fired: {fut.result()['endpoint']}")
    )

# wait() — block until specific futures complete
from concurrent.futures import wait, FIRST_COMPLETED, ALL_COMPLETED
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(time.sleep, d) for d in [0.3, 0.1, 0.2]]
    done, pending = wait(futures, return_when=FIRST_COMPLETED)
    print(f"\nFirst completed: {len(done)}, still pending: {len(pending)}")

---

## 6. `multiprocessing` — True CPU Parallelism

In [ ]:
import multiprocessing as mp
import time

def cpu_intensive(n: int) -> int:
    """Genuinely CPU-bound: can't be sped up by threading (with GIL)."""
    return sum(i * i for i in range(n))


if __name__ == '__main__':   # REQUIRED on Windows / macOS to prevent re-spawning
    N = 3_000_000
    TASKS = [N] * 4

    # Sequential
    t0 = time.perf_counter()
    results = [cpu_intensive(n) for n in TASKS]
    seq = time.perf_counter() - t0

    # Parallel processes — each has its OWN Python interpreter, NO GIL issue
    t0 = time.perf_counter()
    with mp.Pool(processes=4) as pool:
        results = pool.map(cpu_intensive, TASKS)
    par = time.perf_counter() - t0

    print(f"Sequential:  {seq:.2f}s")
    print(f"4 Processes: {par:.2f}s")
    print(f"Speedup:     {seq/par:.1f}x")
    print(f"CPU cores:   {mp.cpu_count()}")

In [ ]:
# ── Process creation methods ───────────────────────────────────────────────
# 'spawn'  — fresh Python interpreter (default on Windows, macOS)
#            safest but slowest — no inherited file handles
# 'fork'   — copy parent process (default on Linux)
#            fast but unsafe with threads (can deadlock)
# 'forkserver' — separate server process does the forking
#                safe with threads

# Set start method (call once, early in the program)
# mp.set_start_method('spawn')   # default on Windows/macOS
# mp.set_start_method('fork')    # default on Linux

# Get current method
print(f"Start method: {mp.get_start_method()}")

# Individual Process object (like threading.Thread)
def process_worker(name: str, result_queue: mp.Queue) -> None:
    import os
    result_queue.put((name, os.getpid()))

if __name__ == '__main__':
    q = mp.Queue()
    procs = [mp.Process(target=process_worker, args=(f"P{i}", q)) for i in range(3)]
    for p in procs: p.start()
    for p in procs: p.join()
    while not q.empty():
        name, pid = q.get()
        print(f"  {name}: PID={pid}")

In [ ]:
# ── Pool methods ──────────────────────────────────────────────────────────
import multiprocessing as mp

def square(x): return x * x
def is_prime(n):
    if n < 2: return False
    return all(n % i != 0 for i in range(2, int(n**0.5)+1))

if __name__ == '__main__':
    with mp.Pool() as pool:
        # map — 1:1 mapping, blocks until all complete
        squares = pool.map(square, range(10))
        print(f"map:       {squares}")

        # imap — lazy iterator (memory-efficient for large inputs)
        squares_iter = list(pool.imap(square, range(10)))
        print(f"imap:      {squares_iter}")

        # starmap — unpack tuples as multiple arguments
        pairs = [(2, 3), (4, 5), (6, 7)]
        sums  = pool.starmap(lambda a, b: a + b, pairs)
        print(f"starmap:   {sums}")

        # apply_async — non-blocking single call
        future = pool.apply_async(square, (42,))
        print(f"apply_async: {future.get()}")

        # map_async — non-blocking map
        result = pool.map_async(is_prime, range(2, 20))
        primes = [n for n, p in zip(range(2,20), result.get()) if p]
        print(f"primes 2-19: {primes}")

---

## 7. `concurrent.futures.ProcessPoolExecutor`

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import time

def factorise(n: int) -> list[int]:
    """Return prime factors of n."""
    factors = []
    d = 2
    while d * d <= n:
        while n % d == 0:
            factors.append(d)
            n //= d
        d += 1
    if n > 1:
        factors.append(n)
    return factors


if __name__ == '__main__':
    numbers = [999999937, 999999929, 999999893, 999999883, 982451653, 982451629]

    t0 = time.perf_counter()
    with ProcessPoolExecutor() as pool:
        futures = {pool.submit(factorise, n): n for n in numbers}
        for f in as_completed(futures):
            n      = futures[f]
            result = f.result()
            print(f"  {n} → {result}")

    print(f"  Total: {time.perf_counter()-t0:.2f}s")

---

## 8. Shared State Between Processes

In [ ]:
import multiprocessing as mp

# Processes do NOT share memory by default.
# Use these mechanisms to share state:

# ── Value and Array (shared memory) ──────────────────────────────────────
def increment_shared(val, arr, n):
    for _ in range(n):
        with val.get_lock():
            val.value += 1
    for i in range(len(arr)):
        with arr.get_lock():
            arr[i] += 1

if __name__ == '__main__':
    shared_val = mp.Value('i', 0)          # 'i' = C int
    shared_arr = mp.Array('d', [0.0]*5)    # 'd' = C double

    procs = [mp.Process(target=increment_shared,
                        args=(shared_val, shared_arr, 1000))
             for _ in range(4)]
    for p in procs: p.start()
    for p in procs: p.join()

    print(f"Shared Value (expected 4000): {shared_val.value}")
    print(f"Shared Array (expected 4000): {list(shared_arr)}")

In [ ]:
# ── Manager — proxy objects backed by a server process ───────────────────
# Supports richer types (list, dict, Queue, Namespace, etc.) but slower.

def worker_with_manager(shared_list, shared_dict, worker_id):
    shared_list.append(worker_id)
    shared_dict[f'worker_{worker_id}'] = worker_id ** 2

if __name__ == '__main__':
    with mp.Manager() as manager:
        shared_list = manager.list()
        shared_dict = manager.dict()

        procs = [mp.Process(target=worker_with_manager,
                            args=(shared_list, shared_dict, i))
                 for i in range(5)]
        for p in procs: p.start()
        for p in procs: p.join()

        print(f"Shared list: {sorted(shared_list)}")
        print(f"Shared dict: {dict(shared_dict)}")

---

## 9. `asyncio` — Cooperative Async I/O

In [ ]:
import asyncio
import time

# asyncio is SINGLE-THREADED. Coroutines voluntarily yield control
# at 'await' points. The event loop then runs another ready coroutine.
# No thread switching overhead, no locks needed (usually).
# Best for: thousands of concurrent I/O operations (HTTP, DB, sockets).

# ── Coroutines — async def functions ─────────────────────────────────────
async def greet(name: str, delay: float) -> str:
    print(f"  {name}: starting")
    await asyncio.sleep(delay)   # yield control — event loop runs others
    print(f"  {name}: done")
    return f"Hello, {name}!"

# Must run coroutines inside an event loop
async def main_basic():
    # Sequential — each awaits before the next starts
    print("Sequential:")
    t0 = time.perf_counter()
    r1 = await greet('Alice', 0.3)
    r2 = await greet('Bob',   0.3)
    print(f"  {time.perf_counter()-t0:.2f}s  (0.6s sequential)")

    # Concurrent with gather — all run at the same time
    print("\nConcurrent with gather:")
    t0 = time.perf_counter()
    results = await asyncio.gather(
        greet('Alice', 0.3),
        greet('Bob',   0.3),
        greet('Carol', 0.3),
    )
    print(f"  {time.perf_counter()-t0:.2f}s  (0.3s concurrent!)")
    print(f"  Results: {results}")

asyncio.run(main_basic())

---

## 10. `async`/`await` Patterns

In [ ]:
import asyncio

# ── Tasks — fire and forget / manage lifecycle ────────────────────────────
async def background_job(name: str) -> str:
    await asyncio.sleep(0.2)
    return f"{name} result"

async def main_tasks():
    # create_task schedules the coroutine to run concurrently
    task1 = asyncio.create_task(background_job('task-1'), name='t1')
    task2 = asyncio.create_task(background_job('task-2'), name='t2')

    print(f"tasks created, doing other work...")
    await asyncio.sleep(0.05)   # yield — tasks can make progress

    r1 = await task1
    r2 = await task2
    print(f"  {r1}, {r2}")

    # cancel a task
    task3 = asyncio.create_task(background_job('task-3'))
    task3.cancel()
    try:
        await task3
    except asyncio.CancelledError:
        print("  task3 was cancelled")

asyncio.run(main_tasks())

In [ ]:
# ── gather vs TaskGroup (3.11+) vs as_completed ───────────────────────────

async def flaky(n: int) -> int:
    await asyncio.sleep(0.1 * n)
    if n == 3:
        raise ValueError(f"Task {n} failed!")
    return n * 10

async def main_gather():
    # gather — waits for all; return_exceptions captures exceptions as values
    results = await asyncio.gather(
        *[flaky(i) for i in range(5)],
        return_exceptions=True   # don't cancel others on failure
    )
    for i, r in enumerate(results):
        if isinstance(r, Exception):
            print(f"  flaky({i}): ERROR — {r}")
        else:
            print(f"  flaky({i}): {r}")

asyncio.run(main_gather())

In [ ]:
# ── TaskGroup (Python 3.11+) — structured concurrency ───────────────────
# All tasks are cancelled if any raises. Errors are reported together.
# The preferred modern approach over gather().

import sys

async def main_taskgroup():
    if sys.version_info < (3, 11):
        print("TaskGroup requires Python 3.11+")
        return

    results = []

    async with asyncio.TaskGroup() as tg:
        async def fetch_and_store(n):
            await asyncio.sleep(0.05 * n)
            results.append(n * 10)

        for i in range(1, 6):
            tg.create_task(fetch_and_store(i))
    # All tasks complete (or all cancelled on exception) before we get here

    print(f"  TaskGroup results: {sorted(results)}")

asyncio.run(main_taskgroup())

In [ ]:
# ── Timeouts ──────────────────────────────────────────────────────────────

async def slow_operation():
    await asyncio.sleep(2.0)
    return "done"

async def main_timeout():
    # asyncio.timeout (3.11+) — context manager approach
    if sys.version_info >= (3, 11):
        try:
            async with asyncio.timeout(0.5):
                result = await slow_operation()
        except asyncio.TimeoutError:
            print("  Timed out after 0.5s (asyncio.timeout)")

    # wait_for — also works in older Python
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=0.5)
    except asyncio.TimeoutError:
        print("  Timed out after 0.5s (wait_for)")

asyncio.run(main_timeout())

In [ ]:
# ── asyncio.Queue — async producer/consumer ───────────────────────────────

async def async_producer(q: asyncio.Queue):
    for i in range(6):
        await asyncio.sleep(0.05)
        await q.put(i)
        print(f"  Produced: {i}")
    await q.put(None)   # sentinel

async def async_consumer(q: asyncio.Queue, cid: int):
    while True:
        item = await q.get()
        if item is None:
            await q.put(None)   # pass sentinel to other consumers
            break
        print(f"  Consumer {cid} got: {item}")
        await asyncio.sleep(0.1)   # simulate processing
        q.task_done()

async def main_queue():
    q = asyncio.Queue(maxsize=3)
    await asyncio.gather(
        async_producer(q),
        async_consumer(q, 1),
        async_consumer(q, 2),
    )

asyncio.run(main_queue())

In [ ]:
# ── asyncio sync primitives ────────────────────────────────────────────────
# asyncio.Lock, asyncio.Semaphore, asyncio.Event, asyncio.Condition
# — same concepts as threading, but for coroutines

async def main_async_lock():
    lock = asyncio.Lock()
    results = []

    async def critical_section(task_id):
        async with lock:
            # Only one coroutine at a time
            results.append(task_id)
            await asyncio.sleep(0.01)

    await asyncio.gather(*[critical_section(i) for i in range(5)])
    print(f"  Ordered results: {results}")

asyncio.run(main_async_lock())

---

## 11. Mixing Concurrency Models

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# ── Run blocking code in asyncio without blocking the event loop ──────────
# loop.run_in_executor() submits work to a thread or process pool
# and wraps the result in a coroutine-compatible Future.

def blocking_io(name: str, duration: float) -> str:
    """Blocking function — cannot be awaited directly."""
    time.sleep(duration)   # would block the event loop if called directly
    return f"{name} done"

def cpu_work(n: int) -> int:
    """CPU-bound work — best in a process pool."""
    return sum(i*i for i in range(n))


async def main_mixed():
    loop = asyncio.get_running_loop()

    # Run blocking I/O in a thread pool — doesn't block the event loop
    with ThreadPoolExecutor(max_workers=4) as tp:
        io_tasks = [
            loop.run_in_executor(tp, blocking_io, f'task-{i}', 0.2)
            for i in range(4)
        ]
        io_results = await asyncio.gather(*io_tasks)
    print(f"I/O results: {io_results}")

    # Run CPU-bound work in a process pool — true parallelism
    with ProcessPoolExecutor(max_workers=2) as pp:
        cpu_tasks = [
            loop.run_in_executor(pp, cpu_work, 1_000_000)
            for _ in range(4)
        ]
        cpu_results = await asyncio.gather(*cpu_tasks)
    print(f"CPU results: {cpu_results[:2]}...")


if __name__ == '__main__':
    asyncio.run(main_mixed())

In [ ]:
# asyncio.to_thread (Python 3.9+) — even simpler interface
import asyncio

def legacy_blocking_call(x: int) -> int:
    """A third-party blocking function we can't make async."""
    import time
    time.sleep(0.1)
    return x * 2

async def main_to_thread():
    # to_thread runs the function in a thread pool automatically
    results = await asyncio.gather(*[
        asyncio.to_thread(legacy_blocking_call, i)
        for i in range(5)
    ])
    print(f"to_thread results: {results}")

asyncio.run(main_to_thread())

---

## 12. Free-Threaded Python — PEP 703, Python 3.13t & 3.14

This is the most significant change to Python's concurrency model in decades.

In [ ]:
# ── History and motivation ────────────────────────────────────────────────
history = """
THE GIL TIMELINE
================

~1992     GIL introduced in CPython for safe reference counting.
2016      Larry Hastings' "GIL-ectomy" talk at Python Language Summit.
2021      Sam Gross (Meta) shares nogil CPython fork — proof of concept.
2023      PEP 703 ("Making the GIL Optional in CPython") accepted by
          the Python Steering Council.
Oct 2024  Python 3.13 releases with an EXPERIMENTAL free-threaded build
          (install with the 't' suffix: python3.13t).
          Single-thread overhead: ~40% slower than 3.13 (GIL build).
2025      PEP 779 accepted — free-threaded build is OFFICIALLY SUPPORTED
          (no longer experimental) starting in Python 3.14.
          Single-thread overhead reduced significantly vs 3.13t.
          Still NOT the default build — the GIL build remains default.
Future    Phase III (PEP 703): free-threaded becomes the DEFAULT.
          Timeline depends on ecosystem adoption.

NOTE: The GIL is not removed — it is OPTIONAL. The standard GIL build
continues to exist and will for the foreseeable future.
"""
print(history)

In [ ]:
# ── Installing free-threaded Python ───────────────────────────────────────
install_guide = """
INSTALLING FREE-THREADED PYTHON
================================

Python 3.14+ (officially supported):
  # macOS with Homebrew
  brew install python@3.14
  # The free-threaded variant is installed alongside the standard build

  # Windows — use python.org installer
  # Select 'Customize installation' → check 'Free-threaded binaries'

  # Ubuntu/Debian (from deadsnakes PPA)
  sudo add-apt-repository ppa:deadsnakes/ppa
  sudo apt install python3.14 python3.14t    # 't' suffix = free-threaded

  # With pyenv
  pyenv install 3.14t
  pyenv shell 3.14t

Python 3.13 (experimental):
  pyenv install 3.13t     # note the 't' suffix
  python3.13t script.py

Verifying free-threaded mode:
  python3 -c "import sys; print(sys._is_gil_enabled())"   # False = GIL off
  python3 -VV    # shows 'experimental free-threading build' in 3.13

Controlling the GIL at runtime:
  PYTHON_GIL=0 python3 script.py    # disable GIL (free-threaded build only)
  PYTHON_GIL=1 python3 script.py    # re-enable GIL even in free-threaded build
  python3 -X gil=0 script.py        # same via command-line flag
"""
print(install_guide)

In [ ]:
# ── What changes internally ───────────────────────────────────────────────
internals = """
HOW FREE-THREADING WORKS INTERNALLY
=====================================

Reference counting:
  The GIL protected CPython's reference counts (ob_refcnt).
  Free-threading uses per-object locks and BIASED REFERENCE COUNTING:
    - An object is 'owned' by the thread that created it
    - Local (same-thread) refcount changes: no locking needed
    - Cross-thread refcount changes: use atomic operations
  This makes single-threaded code slower (more bookkeeping) but
  allows multi-threaded code to truly run in parallel.

Object safety:
  Built-in types (list, dict, set) are individually locked in 3.13.
  Individual operations are safe, but compound operations still need
  explicit synchronisation (see Section 13).

C extensions:
  Extensions that relied on the GIL for thread safety may be UNSAFE
  in free-threaded mode. They must be audited and marked as
  'free-threading-safe' with Py_mod_gil = Py_MOD_GIL_NOT_USED.
  If an unmarked extension is imported, the GIL is automatically
  re-enabled and a warning is printed.

Garbage collection:
  Python 3.14 introduces a thread-safe incremental garbage collector
  that reduces pause times in free-threaded mode.
"""
print(internals)

In [ ]:
# ── Checking GIL status at runtime ───────────────────────────────────────
import sys, sysconfig

def check_free_threading():
    version = sys.version_info
    is_ft_build = bool(sysconfig.get_config_var('Py_GIL_DISABLED'))

    print(f"Python version:         {sys.version.split()[0]}")
    print(f"Free-threaded build:    {is_ft_build}")

    if version >= (3, 13):
        gil_enabled = sys._is_gil_enabled()
        print(f"GIL currently enabled:  {gil_enabled}")

        if not gil_enabled:
            print("Running in FREE-THREADED mode — true CPU parallelism available!")
        elif is_ft_build:
            print("Free-threaded build but GIL is ON (e.g. unsafe extension loaded)")
        else:
            print("Standard GIL build — threading limited for CPU-bound tasks")
    else:
        print("Free-threading not available (requires Python 3.13+)")


check_free_threading()

In [ ]:
# ── What free-threading enables ───────────────────────────────────────────
# Run this script with python3.14t (or python3.13t) to see the speedup.
# With the standard GIL build, both timings will be roughly equal.

import threading, time

def cpu_bound_work(n: int) -> int:
    """Pure Python CPU work. GIL-limited in standard build."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 3_000_000

# Sequential
t0 = time.perf_counter()
cpu_bound_work(N)
cpu_bound_work(N)
cpu_bound_work(N)
cpu_bound_work(N)
seq_time = time.perf_counter() - t0

# Threaded
t0 = time.perf_counter()
threads = [threading.Thread(target=cpu_bound_work, args=(N,)) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
thr_time = time.perf_counter() - t0

print(f"Sequential:  {seq_time:.3f}s")
print(f"4 Threads:   {thr_time:.3f}s")
print(f"Speedup:     {seq_time/thr_time:.2f}x")
print()

is_ft = bool(sysconfig.get_config_var('Py_GIL_DISABLED'))
if is_ft and sys.version_info >= (3, 13) and not sys._is_gil_enabled():
    print("Running FREE-THREADED — expect ~Nx speedup on N cores!")
else:
    print("Running with GIL — expect ~1x speedup (GIL limits CPU threads).")
    print("Try: python3.14t this_script.py  to see the difference.")

---

## 13. Thread Safety in Free-Threaded Python

In [ ]:
# ── What's still safe (atomic operations) ────────────────────────────────
# In free-threaded Python, individual operations on built-in types are
# protected by per-object locks. These are ATOMIC:
#
#   dict:   d[key] = value
#           del d[key]
#           value = d[key]
#
#   list:   lst.append(x)        — safe
#           lst.pop()            — safe
#           lst[i] = x           — safe on existing index
#           x = lst[i]           — safe
#
#   set:    s.add(x)             — safe
#           s.discard(x)         — safe
#
# THESE ARE STILL NOT ATOMIC (need explicit locking):
#   x += 1             — read-modify-write, two separate operations
#   if key in d: d[key] = ...   — check-then-act, not atomic
#   lst[i] += 1        — read-modify-write
#   for x in lst: ...  — list may change during iteration

import threading

# ❌ STILL a race condition in free-threaded Python
counter = 0

def unsafe_ft():
    global counter
    for _ in range(10000):
        counter += 1   # read-modify-write — NOT atomic!

threads = [threading.Thread(target=unsafe_ft) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Unsafe counter (expected 40000): {counter}")
print("(May be less than 40000 even in free-threaded mode — race condition)")

In [ ]:
# ── Thread-safe patterns for free-threaded Python ─────────────────────────

import threading
from threading import Lock

# Pattern 1: Lock around read-modify-write
counter = 0
lock = Lock()

def safe_with_lock():
    global counter
    for _ in range(10000):
        with lock:
            counter += 1

counter = 0
threads = [threading.Thread(target=safe_with_lock) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Lock-protected counter: {counter}")   # always 40000

# Pattern 2: Use queue.Queue — inherently thread-safe
import queue
result_q = queue.Queue()

def worker_queue():
    result_q.put(sum(range(1000)))

threads = [threading.Thread(target=worker_queue) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

total = 0
while not result_q.empty():
    total += result_q.get()
print(f"Queue-based total: {total}")

In [ ]:
# Pattern 3: Immutable data — share freely, no locking needed
from typing import NamedTuple

class Config(NamedTuple):   # NamedTuple is immutable
    host: str
    port: int
    debug: bool = False

# Shared read-only config — safe to pass to any thread
shared_config = Config('localhost', 5432)

def thread_using_config():
    # Reading immutable data is always safe — no locking needed
    return f"{shared_config.host}:{shared_config.port}"

threads = [threading.Thread(target=thread_using_config) for _ in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print("Immutable data shared safely across 10 threads")

# Pattern 4: Thread-local state — each thread has its own copy
local = threading.local()

def thread_with_local_state(tid):
    local.counter = 0
    for _ in range(1000):
        local.counter += 1   # only this thread touches local.counter
    return local.counter

print(f"Thread-local state: each thread gets its own counter")

In [ ]:
# ── Checking extension support for free-threading ─────────────────────────
check_ext = """
# Checking if a C extension supports free-threading:
import importlib
spec = importlib.util.find_spec('numpy')
# If numpy is marked as free-threading safe, the GIL stays disabled.
# If NOT marked, importing it will re-enable the GIL and print a warning:
#   RuntimeWarning: The global interpreter lock (GIL) has been enabled
#   to load module 'numpy', which has not declared that it can run without
#   the GIL. Use PYTHONGIL_DISABLED=1 to silence this warning.

# Track which packages are free-threading compatible:
#   https://py-free-threading.github.io/tracking/
#
# As of 2025, these popular packages have free-threading support:
#   numpy, scipy, pandas (in progress), Cython, pydantic, cryptography,
#   aiohttp, httpx, Pillow, and many more.
"""
print(check_ext)

In [ ]:
# ── Writing thread-safe code for free-threaded Python ─────────────────────

best_practices = """
THREAD SAFETY BEST PRACTICES (free-threaded Python)
=====================================================

SAFE (no locking needed):
  ✅ Reading immutable objects (str, int, float, tuple, frozenset, NamedTuple)
  ✅ Individual dict/list/set operations (append, get, add)
  ✅ Writing to thread-local storage
  ✅ Calling pure functions with no shared state
  ✅ Passing data via queue.Queue

STILL NEEDS LOCKING:
  ⚠️  x += 1 (or any read-modify-write)
  ⚠️  Check-then-act: if key not in d: d[key] = ...
  ⚠️  Iterating a container while another thread modifies it
  ⚠️  Multi-step operations on shared state
  ⚠️  Custom __add__, __setattr__, etc. on your own classes

DESIGN PATTERNS THAT HELP:
  ✅ Prefer immutable data structures
  ✅ Prefer message passing (queue) over shared memory
  ✅ Keep shared state minimal and well-defined
  ✅ Use threading.Lock for critical sections
  ✅ Use concurrent.futures — hides low-level thread management
  ✅ Use dataclasses(frozen=True) for shared config

TOOLS:
  ThreadSanitizer (TSan) — detects data races at runtime
    python3 -X dev script.py
  py-free-threading.github.io — ecosystem compatibility tracker
"""
print(best_practices)

---

## 14. Choosing the Right Model

In [ ]:
decision_guide = """
WHICH CONCURRENCY MODEL?
========================

1. Do you have CPU-bound work?
   │
   ├─ YES, and you're on Python 3.14+ with free-threaded build?
   │    → threading / ThreadPoolExecutor  (true parallelism, lower overhead)
   │
   ├─ YES, standard GIL build?
   │    → multiprocessing / ProcessPoolExecutor
   │    → Or consider numpy/Cython which release the GIL
   │
   └─ NO (I/O-bound) → go to step 2

2. How many concurrent I/O operations?
   │
   ├─ < 100 operations, simple code preferred?
   │    → threading / ThreadPoolExecutor
   │
   ├─ > 100 operations, or already using async libs (aiohttp, asyncpg)?
   │    → asyncio  (lower overhead, scales to thousands)
   │
   └─ Mixing blocking libs with async code?
        → asyncio + asyncio.to_thread() for blocking calls

SUMMARY TABLE:
  ┌───────────────────┬────────────┬────────────┬──────────────────┐
  │ Scenario          │ Standard   │ asyncio    │ Free-threaded    │
  │                   │ threading  │            │ (3.14t)          │
  ├───────────────────┼────────────┼────────────┼──────────────────┤
  │ I/O-bound (few)   │ ✅ Good    │ ✅ Good    │ ✅ Good          │
  │ I/O-bound (1000+) │ ⚠️  Memory │ ✅ Best    │ ✅ Good          │
  │ CPU-bound         │ ❌ GIL     │ ❌ 1-thread│ ✅ Best          │
  │ Mixed I/O + CPU   │ ⚠️  GIL    │ +to_thread │ ✅ Best          │
  │ Simplicity        │ ✅         │ ⚠️  async  │ ✅               │
  │ Library support   │ ✅ Mature  │ ✅ Mature  │ ⚠️  Evolving     │
  └───────────────────┴────────────┴────────────┴──────────────────┘

NOTE: multiprocessing is best for CPU-bound on standard GIL builds,
but has higher overhead (memory, startup time, IPC) than threading.
"""
print(decision_guide)

---

## 15. Common Pitfalls and Debugging

In [ ]:
import threading, time

# ── Pitfall 1: Deadlock ───────────────────────────────────────────────────
# Two threads each hold a lock the other needs → both wait forever.

lock_a = threading.Lock()
lock_b = threading.Lock()

def thread1_deadlock():
    with lock_a:
        time.sleep(0.01)   # yield so thread2 can grab lock_b
        with lock_b:       # waits for lock_b — but thread2 holds it!
            print("thread1 done")

def thread2_deadlock():
    with lock_b:
        time.sleep(0.01)
        with lock_a:       # waits for lock_a — but thread1 holds it!
            print("thread2 done")

# ✅ FIX: always acquire locks in the same order
def thread1_fixed():
    with lock_a:           # always acquire a before b
        with lock_b:
            print("  thread1 done (fixed)")

def thread2_fixed():
    with lock_a:           # same order — no deadlock possible
        with lock_b:
            print("  thread2 done (fixed)")

t1 = threading.Thread(target=thread1_fixed)
t2 = threading.Thread(target=thread2_fixed)
t1.start(); t2.start()
t1.join();  t2.join()

In [ ]:
# ── Pitfall 2: Exception handling in threads ──────────────────────────────
# Exceptions in a thread do NOT propagate to the main thread!

import traceback

def buggy_thread():
    time.sleep(0.1)
    raise ValueError("oops — this exception disappears!")

t = threading.Thread(target=buggy_thread, daemon=True)
t.start()
t.join()
print("Main thread continues — no exception propagated!")

# ✅ FIX 1: Use concurrent.futures — exceptions are re-raised on .result()
from concurrent.futures import ThreadPoolExecutor

def buggy_task():
    raise ValueError("task failed!")

with ThreadPoolExecutor() as pool:
    f = pool.submit(buggy_task)
    try:
        f.result()   # raises the original exception here
    except ValueError as e:
        print(f"Caught from future: {e}")

# ✅ FIX 2: Catch inside the thread and store the error
errors = []
lock_e = threading.Lock()

def safe_thread():
    try:
        raise ValueError("error in thread")
    except Exception as e:
        with lock_e:
            errors.append(e)

t = threading.Thread(target=safe_thread)
t.start(); t.join()
print(f"Stored errors: {errors}")

In [ ]:
# ── Pitfall 3: Daemon threads silently die at program exit ────────────────
# Daemon threads are killed when the main thread exits — no cleanup!

def cleanup_sensitive_work():
    try:
        time.sleep(1.0)   # program might exit before this finishes!
        print("  cleanup done")
    finally:
        print("  finally block: may NOT run in daemon thread at exit")

# Daemon thread — exits with the program, may not finish
daemon = threading.Thread(target=cleanup_sensitive_work, daemon=True)
daemon.start()
# If we exit here, cleanup_sensitive_work won't complete!

# ✅ For important cleanup: use non-daemon thread + join()
# ✅ Or: use atexit.register() for last-resort cleanup
print("Use daemon=False and join() for threads that must complete.")

In [ ]:
# ── Pitfall 4: Lambda/closure capture in threads ──────────────────────────

# ❌ All threads capture the same 'i' variable (loop variable)
results_bad = []
lock_r = threading.Lock()
threads = []

for i in range(5):
    t = threading.Thread(target=lambda: (lock_r.acquire(),
                                          results_bad.append(i),
                                          lock_r.release()))
    threads.append(t)

for t in threads: t.start()
for t in threads: t.join()
print(f"Closure bug (probably all 4s): {sorted(results_bad)}")

# ✅ Fix: use default argument to capture value at creation time
results_good = []

def append_value(val, lst, lk):
    with lk:
        lst.append(val)

threads = [threading.Thread(target=append_value, args=(i, results_good, lock_r))
           for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Fixed (all unique): {sorted(results_good)}")

In [ ]:
# ── Debugging tools ───────────────────────────────────────────────────────
debug_tools = """
DEBUGGING CONCURRENT CODE
==========================

1. threading.enumerate()     — list all live threads
   threading.active_count()  — number of live threads

2. faulthandler — print tracebacks for all threads on SIGABRT or crash:
     import faulthandler
     faulthandler.enable()
   Or from command line:
     python3 -X faulthandler script.py

3. ThreadSanitizer (TSan) — detects data races (Linux/macOS):
   Compile CPython with TSan and run your script.
   Or: python3 -X dev script.py   (enables additional checks)

4. asyncio debugging:
     asyncio.run(main(), debug=True)
   Or: PYTHONASYNCIODEBUG=1 python3 script.py
   Reports coroutines that take too long, unawaited coroutines, etc.

5. concurrent.futures + logging:
     import logging
     logging.basicConfig(level=logging.DEBUG)
   Logs thread pool activity at DEBUG level.

6. For free-threaded Python — check race conditions:
     python3 -X dev -W error script.py
   Turns GIL-re-enable warnings into errors.

7. py-spy / viztracer — profilers that show concurrency timelines:
     pip install viztracer
     viztracer --open myscript.py
"""
print(debug_tools)

---

## 16. Quick Reference

In [ ]:
quick_ref = """
CONCURRENCY QUICK REFERENCE
===========================

threading
  Thread(target=fn, args=(...))  → create thread
  t.start()                      → start
  t.join(timeout=None)           → wait for completion
  t.daemon = True                → killed when main exits
  threading.Lock()               → mutual exclusion
  threading.RLock()              → reentrant lock
  threading.Semaphore(n)         → limit N concurrent holders
  threading.Event()              → signal between threads
  threading.Condition()          → wait for state change
  threading.Barrier(n)           → synchronise N threads
  threading.local()              → thread-local storage
  queue.Queue()                  → thread-safe FIFO

concurrent.futures
  ThreadPoolExecutor(max_workers=N)   → thread pool
  ProcessPoolExecutor(max_workers=N)  → process pool
  executor.submit(fn, *args)          → returns Future
  executor.map(fn, iterable)          → blocking map
  future.result(timeout=N)            → get result (blocks)
  future.done()                       → check if complete
  as_completed(futures)               → yield in completion order
  wait(futures, return_when=...)      → wait for subset

multiprocessing
  mp.Pool(processes=N)          → process pool
  pool.map(fn, iterable)        → parallel map
  pool.starmap(fn, pairs)       → parallel map, tuple args
  mp.Queue()                    → inter-process queue
  mp.Value('i', 0)              → shared int
  mp.Array('d', [...])          → shared array
  mp.Manager()                  → proxy server for richer types

asyncio
  async def fn():               → coroutine function
  await coroutine               → yield control
  asyncio.run(coroutine)        → entry point
  asyncio.create_task(coro)     → schedule concurrent task
  asyncio.gather(*coros)        → run all, collect results
  asyncio.wait_for(coro, t)     → timeout
  asyncio.timeout(t)            → context manager timeout (3.11+)
  async with TaskGroup() as tg: → structured concurrency (3.11+)
  asyncio.sleep(s)              → async sleep
  asyncio.to_thread(fn, *args)  → run blocking fn in thread (3.9+)
  asyncio.Queue()               → async-safe queue
  asyncio.Lock()                → async mutex

free-threading (Python 3.13t / 3.14)
  sys._is_gil_enabled()         → check if GIL is active
  PYTHON_GIL=0 python3 ...      → disable GIL at startup
  python3 -X gil=0 ...          → same via flag
  sysconfig.get_config_var('Py_GIL_DISABLED')  → check build type
"""
print(quick_ref)

In [ ]:
# 🏆 Practice: async web scraper simulation
# Demonstrates asyncio + Semaphore for rate limiting + error handling

import asyncio, random, time

async def fetch_page(session_id: int, url: str, sem: asyncio.Semaphore) -> dict:
    """Simulate fetching a URL with rate limiting and occasional failures."""
    async with sem:   # limit concurrent requests
        delay = random.uniform(0.05, 0.3)
        await asyncio.sleep(delay)

        # Simulate occasional failures
        if random.random() < 0.15:
            raise ConnectionError(f"Failed to fetch {url}")

        return {'url': url, 'status': 200, 'size': random.randint(1000, 50000)}


async def scrape(urls: list[str], max_concurrent: int = 5) -> dict:
    """Scrape multiple URLs concurrently with rate limiting and retries."""
    sem      = asyncio.Semaphore(max_concurrent)
    results  = {}
    failures = []
    total_bytes = 0

    async def fetch_with_retry(url: str, retries: int = 2) -> None:
        nonlocal total_bytes
        for attempt in range(retries + 1):
            try:
                result = await fetch_page(id(asyncio.current_task()), url, sem)
                results[url] = result
                total_bytes += result['size']
                return
            except ConnectionError as e:
                if attempt == retries:
                    failures.append({'url': url, 'error': str(e)})
                else:
                    await asyncio.sleep(0.1 * (attempt + 1))  # backoff

    t0 = time.perf_counter()

    async with asyncio.TaskGroup() as tg:
        for url in urls:
            tg.create_task(fetch_with_retry(url))

    elapsed = time.perf_counter() - t0

    return {
        'successful': len(results),
        'failed':     len(failures),
        'total_bytes': total_bytes,
        'elapsed_s':   round(elapsed, 3),
        'failures':    failures,
    }


urls = [f"https://example.com/page/{i}" for i in range(20)]

random.seed(42)
summary = asyncio.run(scrape(urls, max_concurrent=5))

print("=" * 40)
print("SCRAPE SUMMARY")
print("=" * 40)
print(f"URLs attempted:  {len(urls)}")
print(f"Successful:      {summary['successful']}")
print(f"Failed:          {summary['failed']}")
print(f"Total bytes:     {summary['total_bytes']:,}")
print(f"Elapsed:         {summary['elapsed_s']}s")
if summary['failures']:
    print("Failures:")
    for f in summary['failures']:
        print(f"  {f['url']}: {f['error']}")